In [ ]:
from langchain_community.llms import Ollama
llm = Ollama(
    model = "llama3"
)

response = llm.invoke(
    "what is the capital of france"
)

response

'The capital of France is Paris.'

In [15]:
from langchain_groq import ChatGroq 
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

model = ChatGroq(groq_api_key = groq_api_key , model="Llama3-8b-8192")

In [21]:
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate , PromptTemplate 

template1 = PromptTemplate(
    template="write a detailed report on {topic}",
    input_variables=['topic']
)

template2 = PromptTemplate(
    template="write a 5 line summary on the following text {text}",
    input_variables=['text']
)

parser = StrOutputParser()
parser2 = JsonOutputParser()

chain = template1 | model | parser | template2 | model | parser

result = chain.invoke(
    {'topic':'black hole'}
)
print(result)

Here is a 5-line summary of the text:

Black holes are regions in space where gravity is so strong that nothing, not even light, can escape. They are formed when a massive star collapses in on itself, creating a singularity with infinite curvature. Black holes have several properties, including mass, spin, charge, and an event horizon, which marks the boundary beyond which nothing can escape. The behavior of black holes is influenced by their properties, and they can exhibit phenomena such as gravitational lensing and Hawking radiation. Ongoing research is helping to better understand black holes, which continue to fascinate scientists and the public alike.


In [33]:
template3 = PromptTemplate(
    template='give the name , age and city of the fictional person \n {format_instruction}',
    input_variables=[],
    partial_variables={'format_instruction':parser2.get_format_instructions()} # runtime par fill nhi hoga phele se ho jayega thats why partial he 
)

prompt = template3.format()

print(prompt)


give the name , age and city of the fictional person 
 Return a JSON object.


In [34]:
result = model.invoke(prompt)
print(parser2.parse(result.content))

{'name': 'Evelyn Stone', 'age': 32, 'city': 'San Francisco'}


In [50]:
parser = JsonOutputParser()

template = PromptTemplate(
    template="give me 5 facts about {topic} \n {format_instruction}",
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()} # runtime par fill nhi hoga phele se ho jayega thats why partial he 
)


chain = template | model | parser

result = chain.invoke(
    {'topic': "anime"}
)

print(result)

{'fact1': {'title': ' Origins of Anime', 'text': "Anime originated in Japan in the early 20th century, with the first anime film being 'Katsudō Shashin' in 1907."}, 'fact2': {'title': 'Unique Art Style', 'text': 'Anime has a distinct art style that is characterized by vibrant colors, exaggerated facial expressions, and dynamic action sequences.'}, 'fact3': {'title': 'Popularity Worldwide', 'text': 'Anime has become incredibly popular worldwide, with fans from all over the globe enjoying Japanese animation through streaming services, DVDs, and TV broadcasts.'}, 'fact4': {'title': 'Influence on Western Animation', 'text': 'Anime has had a significant influence on Western animation, with many Western animators and studios drawing inspiration from Japanese animation techniques and storytelling styles.'}, 'fact5': {'title': 'Variety of Genres', 'text': 'Anime covers a wide range of genres, including action, comedy, drama, fantasy, horror, romance, science fiction, and more, offering somethi

# structured output Parser

In [54]:
from langchain.output_parsers import StructuredOutputParser , ResponseSchema

schema = [
    ResponseSchema(name = 'fact_1' , description='fact 1 about the topic'),
    ResponseSchema(name = 'fact_2' , description='fact 2 about the topic'),
    ResponseSchema(name = 'fact_3' , description='fact 3 about the topic'),
    ResponseSchema(name = 'fact_4' , description='fact 4 about the topic'),
    ResponseSchema(name = 'fact_5' , description='fact 5 about the topic'),
    ResponseSchema(name = 'fact_6' , description='fact 6 about the topic')
]

parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(
    template="give me 6 facts about {topic} \n {format_instruction}",
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()} # runtime par fill nhi hoga phele se ho jayega thats why partial he 
)

prompt = template.invoke(
    {'topic' : 'anime'}
)

result = model.invoke(prompt)
final_result = parser.parse(result.content)
print(final_result)



{'fact_1': "The word 'anime' is a Japanese pronunciation of the French word 'animation', and it's used to describe Japanese animated television shows and films.", 'fact_2': "The first anime ever made was 'Katsudō Shashin', a short film created in 1907 by Ōten Shimokawa, a Japanese filmmaker.", 'fact_3': 'The most popular anime genre is Shonen, which is targeted towards a male audience and typically features action-packed stories, strong protagonists, and epic battles.', 'fact_4': "The longest-running anime series is 'Sazae-san', which has been on the air since 1969 and has over 7,000 episodes.", 'fact_5': "The most popular anime character is Pikachu, the electric mouse from the 'Pokémon' franchise, which has become a global phenomenon.", 'fact_6': "The highest-grossing anime film of all time is 'Spirited Away', directed by Hayao Miyazaki and released in 2001, which won several awards, including the Academy Award for Best Animated Feature."}


In [55]:
chain = template | model | parser 

response = chain.invoke(
    {'topic' : 'anime'}
)

print(response)

{'fact_1': "Anime is short for 'animation' in Japanese, and the term has been used to describe Japanese animation since the 1970s.", 'fact_2': "The first anime film, 'Katsudō Shashin', was created in 1907 by Ōten Shimokawa, a Japanese filmmaker.", 'fact_3': 'The average length of an anime episode is around 22-24 minutes, although some episodes can range from 10-60 minutes or more.', 'fact_4': 'There are many different genres of anime, including action, comedy, drama, fantasy, horror, romance, and science fiction, among others.', 'fact_5': 'The largest anime convention in the world is the Anime Expo, which is held annually in Los Angeles, California, and attracts over 100,000 attendees.', 'fact_6': 'The most popular anime streaming platform is Crunchyroll, which has over 2 million subscribers and offers a vast library of anime shows and movies.'}


# pydantic outputparser  

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq 
import os 
from dotenv import load_dotenv
from pydantic import Field , BaseModel
from langchain.output_parsers import PydanticOutputParser
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
model = ChatGroq(groq_api_key = api_key , model="Llama3-8b-8192")

class person(BaseModel):
    name : str = Field(description='Name of the person')
    age : int = Field(gt=18 , description='age of the person')
    city : str = Field(description='city name of the city , the person belongs to')

parser = PydanticOutputParser(pydantic_object=person)

template = PromptTemplate(
    template='generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

prompt = template.invoke({'place' : 'russian'})

result = model.invoke(prompt)

final_result = parser.parse(result.content)
print(final_result)

chain = template | model | parser 
final_result = chain.invoke({'place': 'indian'})
print(final_result)

name='Александра Петрова' age=32 city='Москва'
name='Aarav Mehta' age=25 city='Mumbai'


# chains

In [2]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()
import os 

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key , model = "Llama3-8b-8192")

prompt1 = PromptTemplate(
    template="generate a detail report on {topic}",
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

parser = StrOutputParser()

chain = prompt1 | llm | parser | prompt2 | llm | parser

result = chain.invoke({"topic" : "unemployment in India"})
print(result)

Here is a 5-point summary of the report on unemployment in India:

**Point 1: Unemployment Rate in India**
The overall unemployment rate in India stands at 6.1%, with the youth (15-29 years) being the most affected, with an unemployment rate of 12.6%. The unemployment rate among women (15-29 years) is even higher, at 15.5%.

**Point 2: Causes of Unemployment**
The main causes of unemployment in India are lack of job creation, demographic dividend, lack of skilled workforce, informal sector, seasonal unemployment, lack of infrastructure, and regulatory barriers.

**Point 3: Consequences of Unemployment**
Unemployment has severe consequences, including poverty, social unrest, decreased productivity, and brain drain. It also leads to a decrease in overall economic growth and development.

**Point 4: Solutions to Unemployment**
The report suggests a multi-faceted approach to address unemployment, including job creation, skilling and reskilling, informal sector formalization, infrastructure

In [5]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain.schema.runnable import RunnableBranch , RunnableParallel , RunnableLambda
from pydantic import BaseModel , Field
from typing import Literal
from dotenv import load_dotenv
load_dotenv()
import os 

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key , model = "Llama3-8b-8192")

class Feedback(BaseModel):
    sentiment : Literal['positive' , 'negative'] = Field(description="give the sentiment of the feedback")

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template="classify the sentiment of the following feedback text into positive and negative \n {feedback} \n {format_instruction}",
    input_variables=['feedback'],
    partial_variables={'format_instruction' : parser2.get_format_instructions()}
)

prompt2 = PromptTemplate(
    template="write and appropriate response to the positive feedback \n {feedback}",
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template="write and appropriate response to the negative feedback \n {feedback}",
    input_variables=['feedback']
)

classifier_chain = prompt1 | llm | parser2
# result = classifier_chain.invoke({
#     "feedback" : "this is the worst product i ve ever purchased"
# }).sentiment
branch_chain = RunnableBranch(
    (lambda x : x.sentiment == "positive" , prompt2|llm|parser2),
    (lambda x : x.sentiment == "negative", prompt3|llm|parser2),
    RunnableLambda(lambda x : "couldn't find the sentiment") # not a chain hence using runnable lambda
)

chain = classifier_chain | branch_chain

chain.invoke({
    "feedback" : "this is a terrible phone"
})

OutputParserException: Invalid json output: Here is an example of an appropriate response to negative feedback with a sentiment of 'negative':

**Apologize and Acknowledge**

Dear [Customer],

Thank you for taking the time to share your concerns about your recent [experience/purchase] with us. We apologize for any frustration or disappointment we may have caused. We understand that your experience did not meet your expectations, and for that, we are truly sorry.

**Listen and Acknowledge Specific Issues**

We want to assure you that we take all feedback seriously and are committed to making things right. We have reviewed your specific concerns and are taking immediate action to address the issues you've raised. In particular, we are [specific action being taken, e.g. "investigating the matter further" or "offering a refund/exchange"].

**Show Empathy and Understanding**

We can only imagine how frustrating it must have been to encounter [specific issue]. We want to assure you that we v

# runnables

In [15]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
load_dotenv()
import os 

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key , model = "Llama3-8b-8192")


prompt = PromptTemplate(
    input_variables=['topic'],
    template="suggest catchy blog titles about {topic}."
)

chain = LLMChain(llm = llm , prompt = prompt)

output = chain.run("space aircraft") # topic 
print("generated blog title:" , output)


generated blog title: Here are some catchy blog title ideas about space aircraft:

1. **"Reaching for the Stars: The Future of Space Travel"**
2. **"Spacecraft Showdown: The Next Generation of Spaceplanes"**
3. **"Blast Off! The Most Incredible Spacecraft of All Time"**
4. **"Gravity Defied: The Thrilling World of Space Tourism"**
5. **"Spacebound and Beyond: The Evolution of Space Exploration"**
6. **"Rocket Science: How Spacecraft are Changing the Game"**
7. **"Astronauts and Aerospace: The Pioneers of Space Travel"**
8. **"Destination Space: The Ultimate Guide to Spacecraft"**
9. **"Spacecraft Secrets: Uncovering the Technology Behind Space Exploration"**
10. **"Into Orbit: The Incredible Journey of Spacecraft"**
11. **"Space Age: The Future of Human Spaceflight"**
12. **"Flying High: The Amazing Capabilities of Spacecraft"**
13. **"The Cosmos Unlocked: How Spacecraft are Unlocking the Secrets of Space"**
14. **"Spacecraft Smarts: The Intelligent Design Behind Space Exploration"**
1

# runnable sequence 

In [8]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableSequence
load_dotenv()
import os 

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key , model = "Llama3-8b-8192")

prompt = PromptTemplate(
    template='write a joke about {topic}',
    input_variables=['topic']
)

prompt_ = PromptTemplate(
    template="explain the following joke - {text}",
    input_variables=['text']
)

parser = StrOutputParser()

chain = RunnableSequence(prompt , llm , parser , prompt_ , llm , parser)

chain.invoke(
    {'topic' : 'ai'}
)

'A classic tech-related pun!\n\nHere\'s the breakdown:\n\n* "Why did the AI go on a diet?" - This is the setup, asking why an Artificial Intelligence (AI) would want to go on a diet.\n* "Because it wanted to lose some bytes!" - This is the punchline, explaining the reason why the AI went on a diet.\n\nThe wordplay is in the use of "bytes" instead of "pounds" or "kilograms", which are common units of measurement for weight loss. "Bytes" is a unit of digital information, often used to measure the size of files or data storage. In this joke, the AI wants to "lose some bytes", implying it wants to reduce its digital size or memory allocation, rather than its physical weight.\n\nThe joke relies on a play on words, combining the technical concept of bytes with the common goal of dieting (weight loss), creating a humorous and clever connection between the two.'

# runnable parralel

In [16]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableParallel , RunnableSequence
load_dotenv()
import os 

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key , model = "Llama3-8b-8192")

prompt = PromptTemplate(
    template='write a linkedin post on following Topic:{topic}',
    input_variables=['topic']
)

prompt_ = PromptTemplate(
    template="explain a tweet on  following Topic:{topic}",
    input_variables=['topic']
)

parser = StrOutputParser()

parallel_chain= RunnableParallel({
    "tweet": RunnableSequence(prompt , llm , parser),
    "linkedin":RunnableSequence(prompt_,llm,parser)
})

parallel_chain.invoke(
    {'topic' : 'ai'}
)

{'tweet': 'Here\'s a sample LinkedIn post on the topic of AI:\n\n**Title:** "The Future is Here: How AI is Revolutionizing Industries and Transforming the Way We Work"\n\n**Text:**\n\nAs AI continues to evolve and mature, it\'s becoming increasingly clear that its impact on industries and workplaces will be profound. From automating mundane tasks to augmenting human decision-making, AI is transforming the way we work and interact with each other.\n\nAt [Your Company], we\'re at the forefront of this AI revolution, and I\'m excited to share some of the ways we\'re leveraging this technology to drive innovation and growth.\n\n**AI in Action:**\n\n* **Predictive Maintenance**: Our AI-powered predictive maintenance solution is helping manufacturing companies reduce downtime and improve equipment efficiency by up to 30%.\n* **Personalized Customer Experience**: Our AI-driven customer service platform is enabling businesses to deliver tailored experiences to their customers, resulting in inc